# Common generalized Littlewood–Richardson coefficients — optimized SageMath notebook

For partitions $\lambda,\mu$ and $0\le k\le\min(|\lambda|,|\mu|)$, this notebook computes

$$S_k^{\lambda,\mu}(\lambda',\mu') = \sum_{|\delta|+|\gamma|=k} N^\lambda_{\delta,\gamma,\lambda'}N^\mu_{\delta,\gamma,\mu'}.$$

The sums are **over ordered pairs** $(\delta,\gamma)$, including the empty partition. The notebook preserves your original multiplicities and produces LaTeX tables. It avoids enumerating all $\lambda',\mu',\delta,\gamma$ combinations by using `lrcalc.coprod` and a reusable bilinear kernel. It requires a **SageMath kernel**, not ordinary Python.

### Mathematical reorganization

By associativity,
$$N^\lambda_{\delta,\gamma,\lambda'} = \sum_{\nu\vdash k}c^\nu_{\delta,\gamma}c^\lambda_{\nu,\lambda'}.$$
Define the symmetric kernel
$$K_k(\nu,\omega)=\sum_{|\delta|+|\gamma|=k}c^\nu_{\delta,\gamma}c^\omega_{\delta,\gamma}.$$
Then
$$\boxed{S_k^{\lambda,\mu}(\lambda',\mu')=\sum_{\nu,\omega\vdash k}c^\lambda_{\nu,\lambda'}K_k(\nu,\omega)c^\mu_{\omega,\mu'}}.$$

`coprod(λ, all=1)` returns the sparse coefficients $c^\lambda_{\nu,\lambda'}$ for all ordered pairs, so we use those entries directly. The kernel depends only on $(\nu,\omega)$ and can be reused across rows and tables.

In [75]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import sage.libs.lrcalc.lrcalc as lrcalc

def part(p):
    p = tuple(int(x) for x in p)
    if any(x <= 0 for x in p) or any(a < b for a, b in zip(p, p[1:])):
        raise ValueError(f"Invalid partition: {p}")
    return p

def degree(p): return sum(p)

@lru_cache(maxsize=None)
def lr(outer, left, right):
    outer, left, right = part(outer), part(left), part(right)
    if degree(outer) != degree(left) + degree(right): return 0
    return int(lrcalc.lrcoef_unsafe(outer, left, right))

@lru_cache(maxsize=None)
def coproduct(p):
    """Sparse map (left, right) -> c^p_{left,right}; all *ordered* pairs."""
    p = part(p)
    return {(part(left), part(right)): int(coef)
            for (left, right), coef in lrcalc.coprod(p, all=1).items()}

@lru_cache(maxsize=None)
def triple_lr(left, middle, right, outer):
    """N^outer_{left,middle,right} using one cached Schur product."""
    left, middle, right, outer = map(part, (left, middle, right, outer))
    if degree(left) + degree(middle) + degree(right) != degree(outer): return 0
    return sum(int(c) * lr(outer, part(nu), right)
               for nu, c in lrcalc.mult(left, middle).items())

print('SageMath LR routines ready')

SageMath LR routines ready


## Kernel and row construction

For fixed $k$, the only potentially nonzero terms have $|\lambda'|=|\lambda|-k$ and $|\mu'|=|\mu|-k$. We build sparse maps indexed by the right-hand partition, rather than checking all partitions of those degrees.

All coproduct multiplicities remain exact integers; no approximation is introduced.

In [76]:
@lru_cache(maxsize=None)
def coproduct_by_right(p, k):
    """Map p_right -> {p_left: c^p_{p_left,p_right}} for |p_left|=k."""
    grouped = defaultdict(dict)
    for (left, right), coeff in coproduct(part(p)).items():
        if degree(left) == k:
            grouped[right][left] = coeff
    return dict(grouped)

@lru_cache(maxsize=None)
def kernel(nu, omega):
    """Sum_{d,g} c^nu_{d,g} c^omega_{d,g}, with ordered (d,g)."""
    nu, omega = part(nu), part(omega)
    if degree(nu) != degree(omega): return 0
    # dot product of two sparse coproduct vectors
    a, b = coproduct(nu), coproduct(omega)
    if len(a) > len(b): a, b = b, a
    return sum(value * b.get(pair, 0) for pair, value in a.items())

@lru_cache(maxsize=None)
def row_data(k, lam, mu):
    """Return {(lam_prime, mu_prime): multiplicity} for one k."""
    lam, mu = part(lam), part(mu)
    if not 0 <= k <= min(degree(lam), degree(mu)): return {}
    A = coproduct_by_right(lam, k)
    B = coproduct_by_right(mu, k)
    if not A or not B: return {}
    all_nu = sorted({nu for terms in A.values() for nu in terms})
    all_omega = sorted({omega for terms in B.values() for omega in terms})
    K = {(nu, omega): kernel(nu, omega) for nu in all_nu for omega in all_omega}
    result = {}
    for lp, left_terms in A.items():
        for mp, right_terms in B.items():
            total = sum(ca * cb * K[(nu, omega)]
                        for nu, ca in left_terms.items()
                        for omega, cb in right_terms.items())
            if total:
                result[(lp, mp)] = total
    return result


def max_degree(lam, mu):
    """Largest k having a nonzero row; not necessarily every smaller row is nonzero."""
    return max((k for k in range(min(degree(lam), degree(mu)) + 1)
                if row_data(k, part(lam), part(mu))), default=0)

print('Kernel and row functions ready')

Kernel and row functions ready


## Table formatting and compilable LaTeX output

`table_tex()` still returns a single table fragment. Both `write_table()` and `generate_tables()` now write **complete standalone LaTeX documents by default**, using the supplied article preamble, author, title, `\maketitle`, `\renewcommand{\arraystretch}{1.6}`, and the matching `table[H]` environment.

Set `standalone=False` to export fragments for an existing LaTeX document. The LR computations and row selection are unchanged.


In [77]:
# Standalone LaTeX document configuration. Change these values as needed.
LATEX_AUTHOR = "Pablo Santos"
LATEX_TITLE = None  # Auto-generated as "DegN partitions" by generate_tables().

LATEX_PREAMBLE = r"""\documentclass[a4paper,12pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage{amsmath,amssymb,amsthm}
\usepackage{bm}
\usepackage{enumitem}
\usepackage{listings}
\usepackage{xcolor}
\usepackage{hyperref}
\usepackage{cleveref}
\usepackage{booktabs}
\usepackage{array}
\usepackage{graphicx}
\usepackage{float}
\usepackage{caption}
\usepackage{subcaption}
\usepackage{authblk}
\usepackage{tabularx}
"""

def latex_part(p):
    p = part(p)
    return r"\varnothing" if not p else "(" + ",".join(map(str, p)) + ")"

def direct_sum(k, lam, mu):
    data = row_data(k, part(lam), part(mu))
    terms = []
    for (lp, mp), multiplicity in sorted(data.items(), reverse=True):
        prefix = "" if multiplicity == 1 else str(multiplicity)
        terms.append(prefix + r"V^{" + latex_part(lp) + "," + latex_part(mp) + "}")
    return r" \oplus ".join(terms) if terms else "0"

def table_tex(lam, mu, include_zeros=False):
    """One LaTeX table (fragment); coefficients and summation unchanged."""
    lam, mu = part(lam), part(mu)
    kmax = min(degree(lam), degree(mu))
    rows = []
    for k in range(kmax, -1, -1):
        expression = direct_sum(k, lam, mu)
        if include_zeros or expression != "0":
            rows.append(rf"$k = {k}: \quad {expression}$ \\ \hline")
    title = rf"Table for $\lambda = {latex_part(lam)}$, $\mu = {latex_part(mu)}$:"
    return "\n".join([
        r"\begin{table}[H]", r"\centering",
        rf"\caption{{{title}}}",
        r"\begin{tabular}{|>{\centering\arraybackslash}p{15cm}|}",
        r"\hline", *rows,
        r"\end{tabular}", r"\end{table}", ""
    ])

def document_start(title, author=LATEX_AUTHOR):
    """Complete standalone LaTeX header, including title and document start."""
    # Escape only text for LaTeX metadata, not math expressions in table rows.
    title = str(title).replace("&", r"\&").replace("%", r"\%")
    author = str(author).replace("&", r"\&").replace("%", r"\%")
    return (LATEX_PREAMBLE + "\n" +
            rf"\title{{{title}}}" + "\n" +
            rf"\author{{{author}}}" + "\n" +
            r"\date{\today}" + "\n\n" +
            r"\begin{document}" + "\n" +
            r"\maketitle" + "\n" +
            r"\renewcommand{\arraystretch}{1.6}" + "\n\n")

def document_end():
    return "\n\\end{document}\n"

def write_table(lam, mu, filename, append=False, include_zeros=False,
                standalone=True, title=None, author=LATEX_AUTHOR):
    """Write a standalone, compilable .tex file by default.

    Set standalone=False for a table fragment. If append=True and standalone=True,
    insert the table before an existing \\end{document} when present.
    """
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    body = table_tex(lam, mu, include_zeros=include_zeros)
    title = title or LATEX_TITLE or f"Deg{max(degree(lam), degree(mu))} partitions"
    if append and path.exists():
        old = path.read_text(encoding="utf-8")
        if standalone:
            marker = r"\end{document}"
            if marker not in old:
                raise ValueError("Existing file lacks \\end{document}; cannot append safely")
            old = old.rsplit(marker, 1)[0]
            path.write_text(old + body + document_end(), encoding="utf-8")
        else:
            with path.open("a", encoding="utf-8") as f:
                f.write(body + "\n")
    else:
        text = document_start(title, author) + body + document_end() if standalone else body
        path.write_text(text, encoding="utf-8")
    return path

# Example: export one complete, compilable .tex document
# write_table((), (), 'tex_tables/single_table.tex')


## Small-degree verification against the original definition

These checks evaluate the defining sum explicitly for small partitions and compare it with the factored formula. They also check the example from your notebook. Validation is important because a misplaced symmetry factor of 2 would change every multiplicity.

In [78]:
@lru_cache(maxsize=None)
def partitions_of(n, maxpart=None):
    n = int(n)
    if n == 0: return ((),)
    cap = n if maxpart is None else min(n, maxpart)
    return tuple((first,) + rest
                 for first in range(cap, 0, -1)
                 for rest in partitions_of(n - first, first))

def reference_coefficient(k, lam, lp, mu, mp):
    """Direct implementation of the original defining sum, for testing only."""
    lam, lp, mu, mp = map(part, (lam, lp, mu, mp))
    if degree(lam) - degree(lp) != k or degree(mu) - degree(mp) != k: return 0
    return sum(triple_lr(d,g,lp,lam)*triple_lr(d,g,mp,mu)
               for i in range(k+1)
               for d in partitions_of(i)
               for g in partitions_of(k-i))

assert triple_lr((1,1), (1,1), (1,1), (3,3)) == 1
assert row_data(0, (2,1), (2,)) == {((2,1),(2,)): 1}
assert row_data(12, (5,3,2,2), (4,4,4)).get(((),())) == 18

for lam, mu in [((2,1),(2,1)), ((3,1),(2,2)), ((2,2),(3,1))]:
    for k in range(min(degree(lam), degree(mu)) + 1):
        fast = row_data(k, lam, mu)
        for lp in partitions_of(degree(lam)-k):
            for mp in partitions_of(degree(mu)-k):
                slow = reference_coefficient(k, lam, lp, mu, mp)
                assert fast.get((lp, mp), 0) == slow, (lam,mu,k,lp,mp,slow)
print('All small-degree checks passed')

All small-degree checks passed


## Batch generation: standalone PDF-ready LaTeX source

`generate_tables(4, 4, "tex_tables/deg4_partitions.tex")` writes one `article` document with **all tables**, the provided package imports, title `Deg4 partitions`, author `Pablo Santos`, today's date, and `\end{document}`. Compile the resulting `.tex` file with `pdflatex` (or on Overleaf).

The optional `standalone=False` produces the old table-fragment format.


In [79]:
def generate_tables(max_lam_degree, max_mu_degree, filename,
                    exploit_symmetry=False, include_zeros=False,
                    standalone=True, title=None, author=LATEX_AUTHOR):
    """Export all requested LaTeX tables to one complete, compilable document.

    To insert just table fragments into another TeX file, use standalone=False.
    """
    max_lam_degree, max_mu_degree = int(max_lam_degree), int(max_mu_degree)
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    doc_title = title or LATEX_TITLE or f"Deg{max(max_lam_degree, max_mu_degree)} partitions"
    with path.open('w', encoding='utf-8') as f:
        if standalone:
            f.write(document_start(doc_title, author))
        for n in range(max_lam_degree + 1):
            for m in range(max_mu_degree + 1):
                if exploit_symmetry and n > m: continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if exploit_symmetry and n == m and lam > mu: continue
                        f.write(table_tex(lam, mu, include_zeros=include_zeros))
                        f.write("\n%--------------------\n")
        if standalone:
            f.write(document_end())
    return path

# Example:
output = generate_tables(7, 7, 'tex_tables/deg4_partitions.tex')
print(output)


tex_tables/deg4_partitions.tex


## Performance notes and next steps

1. `coproduct` is cached, so all sparse LR decompositions are reused.
2. `kernel` is the expensive, but reusable, structural computation. In large batches, it can be precomputed once per degree and shared across target pairs.
3. Grouped coproduct data avoids generating impossible $(\lambda',\mu')$ candidates.
4. When only the **largest** nonzero degree is needed, `max_degree` is sufficient, but this is the largest degree appearing in the full coefficient sum; it is not automatically the **common single-partition expansion** maximum $M(\lambda,\mu)$ discussed earlier.
5. For an investigation of the **common single-partition** problem, a separate routine should compute $\mathcal E(\lambda,\mu)$ using one common $\delta$ rather than a pair $(\delta,\gamma)$.

**Dependencies:** SageMath, with `sage.libs.lrcalc.lrcalc.coprod`, `lrcoef_unsafe`, and `mult`. The code has not been executed in a SageMath runtime here; run the validation cell in your Sage environment before relying on generated tables.

## Top socle layer explorer

This section **reuses the optimized `row_data`, `max_degree`, `coproduct`, and `coproduct_by_right` functions**. It does not change the table-generation algorithm.

For chosen partitions $\lambda,\mu$, the **top nonzero socle layer** is $k_*=\max\{k : \operatorname{row\_data}(k,\lambda,\mu)\ne\varnothing\}$. The explorer lists every $(\lambda',\mu')$ with $S_{k_*}^{\lambda,\mu}(\lambda',\mu')>0$. Select one to inspect **every ordered pair** $(\delta,\gamma)$ with nonzero contribution

$$N^{\lambda}_{\delta,\gamma,\lambda'}\,N^{\mu}_{\delta,\gamma,\mu'}.$$

This reproduces the contributing-pair information printed by the original `CalcSoc`, while additionally showing each factor and its product. The sum of contributions is checked against the optimized multiplicity. Empty partitions are included; $(\delta,\gamma)$ and $(\gamma,\delta)$ are distinguished as ordered pairs, just as in the original algorithm.


In [80]:
from functools import lru_cache
from html import escape
from IPython.display import display, HTML

@lru_cache(maxsize=None)
def highest_nonzero_row(lam, mu):
    """(k, {(lambda_prime, mu_prime): multiplicity}) for the highest nonzero k."""
    lam, mu = part(lam), part(mu)
    for k in range(min(degree(lam), degree(mu)), -1, -1):
        data = row_data(k, lam, mu)
        if data:
            return k, data
    return 0, {}


def top_layer_summands(lam, mu):
    """Sorted top-layer summands as (lambda_prime, mu_prime, multiplicity)."""
    k, data = highest_nonzero_row(part(lam), part(mu))
    return k, [(lp, mp, int(m)) for (lp, mp), m in sorted(data.items())]


@lru_cache(maxsize=None)
def pair_contributions(lam, mu, lp, mp, k=None):
    """All NONZERO ordered (delta,gamma) contributions for one summand.

    Each item is (delta, gamma, left_LR3, right_LR3, product).
    Calculate LR3 factors using the same sparse coproduct decomposition as row_data,
    avoiding a fresh enumeration of all partitions of 0,...,k.
    """
    lam, mu, lp, mp = map(part, (lam, mu, lp, mp))
    if k is None:
        k = highest_nonzero_row(lam, mu)[0]
    k = int(k)
    if degree(lam) - degree(lp) != k or degree(mu) - degree(mp) != k:
        return ()

    left_intermediates = coproduct_by_right(lam, k).get(lp, {})
    right_intermediates = coproduct_by_right(mu, k).get(mp, {})
    if not left_intermediates or not right_intermediates:
        return ()

    # Associativity: N^lam_(delta,gamma,lp)
    # = sum_nu N^lam_(nu,lp) N^nu_(delta,gamma).
    left_factors = defaultdict(int)
    right_factors = defaultdict(int)
    for nu, weight in left_intermediates.items():
        for dg, lr_value in coproduct(nu).items():
            left_factors[dg] += int(weight) * int(lr_value)
    for omega, weight in right_intermediates.items():
        for dg, lr_value in coproduct(omega).items():
            right_factors[dg] += int(weight) * int(lr_value)

    result = []
    for delta, gamma in sorted(left_factors.keys() & right_factors.keys()):
        a, b = left_factors[(delta, gamma)], right_factors[(delta, gamma)]
        if a and b:
            result.append((delta, gamma, int(a), int(b), int(a * b)))
    return tuple(result)


def inspect_top_pair(lam, mu, lp=None, mp=None, *, index=None, k=None, show=True):
    """Inspect a chosen top-layer summand by (lp,mp) OR zero-based index.

    Returns structured results, making it easy to export or test later.
    """
    lam, mu = part(lam), part(mu)
    top_k, all_summands = top_layer_summands(lam, mu)
    if k is None:
        k = top_k
    k = int(k)
    if index is not None:
        if k != top_k:
            raise ValueError('index selection is for the top layer only')
        if not 0 <= int(index) < len(all_summands):
            raise IndexError(f'Choose index from 0 through {len(all_summands)-1}')
        lp, mp, _ = all_summands[int(index)]
    elif lp is None or mp is None:
        raise ValueError('Provide both lp and mp, or provide index=...')
    lp, mp = part(lp), part(mp)
    expected = int(row_data(k, lam, mu).get((lp, mp), 0))
    if not expected:
        raise ValueError('Selected (lambda_prime,mu_prime) is absent from this layer')
    terms = pair_contributions(lam, mu, lp, mp, k)
    total = sum(term[4] for term in terms)
    if int(total) != expected:
        raise AssertionError(f'Contribution sum {total} != row_data multiplicity {expected}')
    result = {'lambda': lam, 'mu': mu, 'k': k, 'lambda_prime': lp,
              'mu_prime': mp, 'multiplicity': expected,
              'contributing_pairs': len(terms), 'terms': terms}
    if show:
        show_pair_contributions(result)
    return result


def show_pair_contributions(result, limit=None):
    """Monochrome notebook display of the chosen summand's contributors."""
    items = result['terms'] if limit is None else result['terms'][:int(limit)]
    p = lambda x: escape(str(x))
    html = ['<div style="font-family:Arial,sans-serif;color:#111">',
            '<h3 style="margin-bottom:5px">Top-layer contributor breakdown</h3>',
            f'<p>k = <b>{result["k"]}</b> &nbsp; '
            f'λ′ = <b>{p(result["lambda_prime"])}</b> &nbsp; '
            f'μ′ = <b>{p(result["mu_prime"])}</b> &nbsp; '
            f'multiplicity = <b>{result["multiplicity"]}</b></p>',
            f'<p>{result["contributing_pairs"]} nonzero ordered (δ, γ) pairs</p>',
            '<div style="overflow-x:auto;max-height:460px;overflow-y:auto">',
            '<table style="border-collapse:collapse;width:100%;font-size:13px";color:#111;background:#111>',
            '<thead><tr style="background:#111;color:white">']
    for h in ('δ','γ','N^λ_(δ,γ,λ′)','N^μ_(δ,γ,μ′)','Product'):
        html.append(f'<th style="padding:9px;border:1px solid #777;text-align:center">{h}</th>')
    html.append('</tr></thead><tbody>')
    for j,(delta,gamma,a,b,c) in enumerate(items):
        bg='#f1f1f1' if j%2 else '#111'
        html.append(f'<tr style="background:{bg}">')
        for x in (p(delta),p(gamma),str(a),str(b),str(c)):
            html.append(f'<td style="padding:7px;border-bottom:1px solid #ddd;text-align:center">{x}</td>')
        html.append('</tr>')
    html += ['</tbody></table></div>',
             f'<p><b>Verified:</b> sum of products = {result["multiplicity"]} '
             '= optimized row multiplicity.</p>', '</div>']
    display(HTML(''.join(html)))


def show_top_layer(lam, mu):
    """Show every (lambda_prime,mu_prime) occurring in the top layer."""
    lam, mu = part(lam), part(mu)
    k, entries = top_layer_summands(lam, mu)
    html = ['<div style="font-family:Arial,sans-serif;color:#111">',
            '<h3>Highest nonzero socle layer</h3>',
            f'<p>λ = <b>{escape(str(lam))}</b>, μ = <b>{escape(str(mu))}</b><br>'
            f'Top layer: <b>k = {k}</b> &nbsp; | &nbsp; '
            f'<b>{len(entries)}</b> distinct (λ′, μ′) summands</p>',
            '<div style="max-height:460px;overflow:auto">',
            '<table style="border-collapse:collapse;width:100%;font-size:13px;color:#111;background:#111">',
            '<thead><tr style="background:#111;color:white">',
            '<th style="padding:9px">Index</th><th>λ′</th><th>μ′</th><th>Multiplicity</th>',
            '</tr></thead><tbody>']
    for idx,(lp,mp,mult) in enumerate(entries):
        bg='#f1f1f1' if idx%2 else 'white'
        html.append(f'<tr style="background:{bg}">'
                    f'<td style="padding:7px;text-align:center">{idx}</td>'
                    f'<td style="text-align:center">{escape(str(lp))}</td>'
                    f'<td style="text-align:center">{escape(str(mp))}</td>'
                    f'<td style="text-align:center">{mult}</td></tr>')
    html += ['</tbody></table></div>',
             '<p style="font-size:12px;color:#555">Select a pair by index below, '
             'or use the interactive dropdown when ipywidgets is available.</p>', '</div>']
    display(HTML(''.join(html)))
    return k, entries


# --- Beta membership checks for a fixed top-layer summand ---

def beta_membership(result, beta):
    """Search this summand's NONZERO ordered (delta, gamma) contributions.

    The supplied result must come from inspect_top_pair. Returns every matching
    pair and its coefficient factors, without recomputing the LR decompositions.
    """
    beta = part(beta)
    matches = []
    for delta, gamma, a, b, product in result['terms']:
        as_delta = delta == beta
        as_gamma = gamma == beta
        if as_delta or as_gamma:
            matches.append({'delta': delta, 'gamma': gamma,
                            'beta_as_delta': as_delta, 'beta_as_gamma': as_gamma,
                            'left_lr3': int(a), 'right_lr3': int(b),
                            'contribution': int(product)})
    return {'beta': beta, 'lambda': result['lambda'], 'mu': result['mu'],
            'lambda_prime': result['lambda_prime'],
            'mu_prime': result['mu_prime'], 'k': result['k'],
            'present_as_delta': any(t['beta_as_delta'] for t in matches),
            'present_as_gamma': any(t['beta_as_gamma'] for t in matches),
            'matching_pairs': len(matches),
            'matching_contribution': sum(t['contribution'] for t in matches),
            'total_multiplicity': result['multiplicity'], 'matches': matches}

def show_beta_membership(report):
    """Display the matching LR pairs, with a monochrome summary."""
    p = lambda x: escape(str(x))
    role = ('both δ and γ' if report['present_as_delta'] and report['present_as_gamma']
            else 'δ' if report['present_as_delta']
            else 'γ' if report['present_as_gamma'] else 'neither δ nor γ')
    html = ['<div style="font-family:Arial,sans-serif;color:#111">',
            '<h3 style="margin-bottom:6px">β membership in the selected summand</h3>',
            f'<p>β = <b>{p(report["beta"])}</b> &nbsp; ' +
            f'λ′ = <b>{p(report["lambda_prime"])}</b> &nbsp; ' +
            f'μ′ = <b>{p(report["mu_prime"])}</b> &nbsp; ' +
            f'k = <b>{report["k"]}</b></p>',
            f'<p><b>Result:</b> β occurs as <b>{role}</b>. ' +
            f'{report["matching_pairs"]} matching ordered pair(s); ' +
            f'total matching contribution: <b>{report["matching_contribution"]}</b> ' +
            f'of <b>{report["total_multiplicity"]}</b>.</p>']
    if report['matches']:
        html.extend(['<div style="max-height:370px;overflow:auto">',
                     '<table style="width:100%;border-collapse:collapse;font-size:13px">',
                     '<thead><tr style="background:#111;color:white">'])
        for label in ('δ', 'γ', 'β position', 'N^λ_(δ,γ,λ′)', 'N^μ_(δ,γ,μ′)', 'Product'):
            html.append(f'<th style="padding:8px;text-align:center">{label}</th>')
        html.append('</tr></thead><tbody>')
        for i, term in enumerate(report['matches']):
            pos = 'both' if term['beta_as_delta'] and term['beta_as_gamma'] else ('δ' if term['beta_as_delta'] else 'γ')
            html.append(f'<tr style="background:{"#f3f3f3" if i%2 else "#111"}">')
            for x in (p(term['delta']), p(term['gamma']), pos, str(term['left_lr3']),
                      str(term['right_lr3']), str(term['contribution'])):
                html.append(f'<td style="padding:7px;text-align:center;border-bottom:1px solid #ddd">{x}</td>')
            html.append('</tr>')
        html.extend(['</tbody></table></div>'])
    html.append('</div>')
    display(HTML(''.join(html)))

def check_beta(lam, mu, beta, lp=None, mp=None, *, index=None, show=True):
    """Public entry point: select top-layer summand, then test β as δ or γ."""
    selected = inspect_top_pair(lam, mu, lp=lp, mp=mp, index=index, show=False)
    report = beta_membership(selected, beta)
    if show:
        show_beta_membership(report)
    return report

def parse_beta(text):
    """Accept Python partition text such as '(3, 2)', '3,2', '[3,2]', or '()'."""
    import ast
    value = ast.literal_eval(text.strip() or '()')
    if isinstance(value, int):
        value = (value,)
    if not isinstance(value, (tuple, list)):
        raise ValueError('Enter a partition, for example (3, 2), [3, 2], or ().')
    parts = tuple(int(x) for x in value)
    if any(x <= 0 for x in parts) or any(a < b for a, b in zip(parts, parts[1:])):
        raise ValueError('β must be a partition: positive parts in weakly decreasing order.')
    return parts


In [95]:
l = (5,)
m = (10,8, 3, 1)

show_top_layer(l,m)

Index,λ′,μ′,Multiplicity
0,(),"(7, 6, 3, 1)",6
1,(),"(7, 7, 2, 1)",6
2,(),"(7, 7, 3)",6
3,(),"(8, 5, 3, 1)",12
4,(),"(8, 6, 2, 1)",18
5,(),"(8, 6, 3)",18
6,(),"(8, 7, 1, 1)",18
7,(),"(8, 7, 2)",24
8,(),"(8, 8, 1)",18
9,(),"(9, 4, 3, 1)",10


(5,
 [((), (7, 6, 3, 1), 6),
  ((), (7, 7, 2, 1), 6),
  ((), (7, 7, 3), 6),
  ((), (8, 5, 3, 1), 12),
  ((), (8, 6, 2, 1), 18),
  ((), (8, 6, 3), 18),
  ((), (8, 7, 1, 1), 18),
  ((), (8, 7, 2), 24),
  ((), (8, 8, 1), 18),
  ((), (9, 4, 3, 1), 10),
  ((), (9, 5, 2, 1), 16),
  ((), (9, 5, 3), 16),
  ((), (9, 6, 1, 1), 18),
  ((), (9, 6, 2), 24),
  ((), (9, 7, 1), 24),
  ((), (9, 8), 6),
  ((), (10, 3, 3, 1), 6),
  ((), (10, 4, 2, 1), 10),
  ((), (10, 4, 3), 10),
  ((), (10, 5, 1, 1), 12),
  ((), (10, 5, 2), 16),
  ((), (10, 6, 1), 18),
  ((), (10, 7), 6)])

### Choose a table and select a summand

Edit `LAMBDA` and `MU` below, then run the cell. If `ipywidgets` is installed, a dropdown lets you select any top-layer `(lambda_prime, mu_prime)` pair and see its contributing `(delta, gamma)` pairs. If widgets are unavailable, use the follow-up cell with `index=...`.

In [81]:
# CHANGE ONLY THESE TWO PARTITIONS
LAMBDA = (4, 2)
MU = (2, 2, 1, 1)

TOP_K, TOP_SUMMANDS = show_top_layer(LAMBDA, MU)

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    options = [
        (f"[{i}] λ′={lp}, μ′={mp}  ·  multiplicity {mult}", i)
        for i, (lp, mp, mult) in enumerate(TOP_SUMMANDS)
    ]
    if options:
        selector = widgets.Dropdown(
            options=options, description='Summand:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='95%'))
        beta_input = widgets.Text(
            value='(3, 2)', description='β:', placeholder='e.g. (3, 2) or ()',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='60%'))
        beta_button = widgets.Button(description='Check β', button_style='')
        details_output = widgets.Output()
        beta_output = widgets.Output()

        def update_details(change=None):
            with details_output:
                clear_output(wait=True)
                inspect_top_pair(LAMBDA, MU, index=int(selector.value))
            # Clear any stale β answer after changing the selected summand.
            with beta_output:
                clear_output(wait=True)

        def update_beta(button=None):
            with beta_output:
                clear_output(wait=True)
                try:
                    beta = parse_beta(beta_input.value)
                    check_beta(LAMBDA, MU, beta, index=int(selector.value))
                except (ValueError, SyntaxError) as err:
                    print(f'Invalid β: {err}')

        selector.observe(update_details, names='value')
        beta_button.on_click(update_beta)
        display(selector, details_output, widgets.HBox([beta_input, beta_button]), beta_output)
        update_details()
    else:
        print('No summands were found.')
except ImportError:
    print('ipywidgets is not installed. Run the manual selection and β-check cells below.')


Index,λ′,μ′,Multiplicity
0,(),(),1


Dropdown(description='Summand:', layout=Layout(width='95%'), options=(('[0] λ′=(), μ′=()  ·  multiplicity 1', 0),), style=DescriptionStyle(description_width='initial'), value=0)

Output()

Output()

### Check a proposed partition β

For the **currently selected** top-layer summand \((\lambda',\mu')\), enter a partition \(\beta\). The checker searches only the nonzero ordered contributor pairs \((\delta,\gamma)\), and reports whether \(\beta=\delta\), \(\beta=\gamma\), or both, including the LR coefficients and matching contribution totals. Use the widget above or the next code cell without `ipywidgets`. A negative result is specific to that selected summand, not to every summand in the top layer.

In [82]:
# Manual selection (works without ipywidgets).
selection = inspect_top_pair(LAMBDA, MU, index=0)

# Check a proposed β against the contributing (δ, γ) pairs of this summand:
BETA = (4,3, 2)
beta_result = beta_membership(selection, BETA)
show_beta_membership(beta_result)

# Or select a different summand directly (0-based index):
# beta_result = check_beta(LAMBDA, MU, beta=(2, 1), index=1)

# Or specify λ′ and μ′ explicitly:
# beta_result = check_beta(LAMBDA, MU, beta=(2, 1), lp=(), mp=())

# Programmatic values:
# beta_result['present_as_delta'], beta_result['present_as_gamma']
# beta_result['matches'], beta_result['matching_contribution']


δ,γ,"N^λ_(δ,γ,λ′)","N^μ_(δ,γ,μ′)",Product
"(2, 1)","(2, 1)",1,1,1


### Consistency checks

The following checks compare the optimized decomposition against the original three-factor LR coefficient formula for the selected summand. They are optional and can be slower than the explorer itself.

In [83]:
# Verify each factor directly through the independent triple_lr routine.
# Set True to run; the explorer already checks that contributions sum to row_data.
VERIFY_SELECTED_FACTORS = False
if VERIFY_SELECTED_FACTORS:
    checked = inspect_top_pair(LAMBDA, MU, index=0, show=False)
    for delta, gamma, a, b, product in checked['terms']:
        assert a == triple_lr(delta, gamma, checked['lambda_prime'], LAMBDA)
        assert b == triple_lr(delta, gamma, checked['mu_prime'], MU)
        assert a*b == product
    print(f"PASS: independently verified {len(checked['terms'])} contributing pairs.")


In [84]:
l = (4,2)
m = (2, 2, 1, 1)


from itertools import zip_longest

def partition_intersection(lam, mu):
    return tuple(
        x for x in (
            min(a, b)
            for a, b in zip_longest(lam, mu, fillvalue=0)
        ) if x > 0
    )

# Manual selection (works without ipywidgets).
selection = inspect_top_pair(l, m, index=0)

# Check a proposed β against the contributing (δ, γ) pairs of this summand:
BETA = partition_intersection(l,m)
print(BETA)


beta_result = beta_membership(selection, BETA)
show_beta_membership(beta_result)

δ,γ,"N^λ_(δ,γ,λ′)","N^μ_(δ,γ,μ′)",Product
"(2, 1)","(2, 1)",1,1,1


(2, 2)


In [85]:

#!!!!!
l = (4,2)
m = (2, 2, 1, 1)

BETA = partition_intersection(l, m)

top_k, summands = show_top_layer(l, m)

found = []

for i in range(len(summands)):
    selection = inspect_top_pair(l, m, index=i, show=False)
    result = beta_membership(selection, BETA)

    if result["present_as_delta"] or result["present_as_gamma"]:
        found.append(i)
        print(f"\nβ found in summand index {i}")
        show_beta_membership(result)

print(f"\nβ = {BETA}")
print(f"Top layer: k = {top_k}")
print(f"Matching summands: {len(found)} / {len(summands)}")


Index,λ′,μ′,Multiplicity
0,(),(),1



β = (2, 2)
Top layer: k = 6
Matching summands: 0 / 1


In [86]:

import csv
import time
from collections import Counter
from IPython.display import display, HTML

def check_all_intersections(n, m, include_empty=True, filename="intersection_results.csv"):
    results = []
    started = time.perf_counter()

    # Same degree convention as generate_tables:
    # all partitions with degrees 0,...,n and 0,...,m.
    lambda_parts = [
        tuple(p)
        for d in range(n + 1)
        for p in Partitions(d)
    ]
    mu_parts = [
        tuple(p)
        for d in range(m + 1)
        for p in Partitions(d)
    ]

    for lam in lambda_parts:
        for mu in mu_parts:
            beta = tuple(partition_intersection(lam, mu))

            if not include_empty and not beta:
                continue

            top_k = int(max_degree(lam, mu))
            summands = row_data(top_k, lam, mu)

            for lp, mp in summands:
                selection = inspect_top_pair(
                    lam, mu, lp=lp, mp=mp, show=False
                )

                check = beta_membership(selection, beta)

                as_delta = bool(check["present_as_delta"])
                as_gamma = bool(check["present_as_gamma"])

                results.append({
                    "lambda": str(lam),
                    "mu": str(mu),
                    "beta": str(beta),
                    "top_k": top_k,
                    "lambda_prime": str(lp),
                    "mu_prime": str(mp),
                    "as_delta": as_delta,
                    "as_gamma": as_gamma,
                    "present": as_delta or as_gamma
                })

    # Export only the compact membership results
    if results:
        with open(filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(results[0]))
            writer.writeheader()
            writer.writerows(results)

    elapsed = time.perf_counter() - started
    counts = Counter(row["present"] for row in results)

    total = len(results)
    present = counts[True]
    absent = counts[False]

    print(f"Partition pairs examined: {len(lambda_parts) * len(mu_parts):,}")
    print(f"Top-layer summands checked: {total:,}")
    print(f"β present: {present:,}")
    print(f"β absent: {absent:,}")
    print(f"Presence rate: {100.0 * int(present) / int(total):.2f}%" if total else "No summands checked")
    print(f"Elapsed: {elapsed:.2f} seconds")
    print(f"Results saved to: {filename}")

    return results


In [87]:
results = check_all_intersections(11, 11)


Partition pairs examined: 38,025
Top-layer summands checked: 75,513
β present: 55,639
β absent: 19,874
Presence rate: 73.68%
Elapsed: 14.85 seconds
Results saved to: intersection_results.csv


# The stronger statement of all summands in the kth max socle filtration contain the intersection

In [88]:
counterexamples = [r for r in results if not r["present"]]

print("Counterexamples:", len(counterexamples))

for r in counterexamples[:200]:
    print(
        "λ =", r["lambda"],
        "μ =", r["mu"],
        "λ ∩ μ =", r["beta"],
        "λ' =", r["lambda_prime"],
        "μ' =", r["mu_prime"],
        "k =", r["top_k"]
    )

Counterexamples: 19874
λ = (2,) μ = (2, 2) λ ∩ μ = (2,) λ' = () μ' = (1, 1) k = 2
λ = (2,) μ = (2, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (2,) k = 2
λ = (2,) μ = (3, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (3,) k = 2
λ = (2,) μ = (2, 2, 1) λ ∩ μ = (2,) λ' = () μ' = (1, 1, 1) k = 2
λ = (2,) μ = (2, 1, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (2, 1) k = 2
λ = (2,) μ = (4, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (4,) k = 2
λ = (2,) μ = (3, 3) λ ∩ μ = (2,) λ' = () μ' = (2, 2) k = 2
λ = (2,) μ = (3, 1, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (3, 1) k = 2
λ = (2,) μ = (2, 2, 2) λ ∩ μ = (2,) λ' = () μ' = (2, 1, 1) k = 2
λ = (2,) μ = (2, 2, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (2, 2) k = 2
λ = (2,) μ = (2, 2, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (1, 1, 1, 1) k = 2
λ = (2,) μ = (2, 1, 1, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (2, 1, 1) k = 2
λ = (2,) μ = (5, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (5,) k = 2
λ = (2,) μ = (4, 1, 1, 1) λ ∩ μ = (2,) λ' = () μ' = (4, 1) k = 2
λ = (2,) μ = (3, 3, 1) λ ∩ μ = (2,) λ' = () μ' = (2, 2, 1) k = 2
λ = (2,) μ = (3, 2, 2)

# The weaker statement of at least one summand in the kth max socle filtration contain the intersection

In [89]:
from collections import defaultdict

grouped = defaultdict(list)

for r in results:
    grouped[(r["lambda"], r["mu"])].append(r["present"])

failures = [
    pair for pair, occurrences in grouped.items()
    if not any(occurrences)
]

print("Partition pairs with no top-layer occurrence of β:", len(failures))
print(failures[:20])

Partition pairs with no top-layer occurrence of β: 2250
[('(4, 2)', '(2, 2, 1, 1)'), ('(4, 2)', '(2, 2, 1, 1, 1)'), ('(4, 2)', '(2, 2, 1, 1, 1, 1)'), ('(4, 2)', '(2, 2, 1, 1, 1, 1, 1)'), ('(4, 2)', '(2, 2, 1, 1, 1, 1, 1, 1)'), ('(4, 2)', '(2, 2, 1, 1, 1, 1, 1, 1, 1)'), ('(4, 1, 1)', '(2, 2, 2)'), ('(3, 3)', '(3, 1, 1, 1)'), ('(3, 3)', '(2, 2, 2)'), ('(3, 3)', '(3, 1, 1, 1, 1)'), ('(3, 3)', '(3, 1, 1, 1, 1, 1)'), ('(3, 3)', '(3, 1, 1, 1, 1, 1, 1)'), ('(3, 3)', '(3, 1, 1, 1, 1, 1, 1, 1)'), ('(3, 3)', '(3, 1, 1, 1, 1, 1, 1, 1, 1)'), ('(3, 1, 1, 1)', '(3, 3)'), ('(2, 2, 2)', '(4, 1, 1)'), ('(2, 2, 2)', '(3, 3)'), ('(2, 2, 2)', '(5, 1, 1)'), ('(2, 2, 2)', '(6, 1, 1)'), ('(2, 2, 2)', '(7, 1, 1)')]


In [90]:

from collections import defaultdict

def show_weak_counterexamples(results, limit=30):
    grouped = defaultdict(list)

    for r in results:
        grouped[(r["lambda"], r["mu"])].append(r)

    counterexamples = []

    for (lam, mu), rows in grouped.items():
        if not any(r["present"] for r in rows):
            counterexamples.append((lam, mu, rows))

    total = len(grouped)
    passed = total - len(counterexamples)
    rate = 100.0 * passed / total if total else 0.0

    print("=" * 75)
    print("WEAK STATEMENT — TOP SOCLE LAYER")
    print("=" * 75)
    print(f"Partition pairs tested: {total}")
    print(f"Pairs satisfying weak statement: {passed}")
    print(f"Counterexamples: {len(counterexamples)}")
    print(f"Weak presence rate: {rate:.2f}%")
    print("=" * 75)

    for i, (lam, mu, rows) in enumerate(counterexamples[:limit], 1):
        print(f"\nCounterexample {i}")
        print(f"  λ     = {lam}")
        print(f"  μ     = {mu}")
        print(f"  β     = {rows[0]['beta']}")
        print(f"  k_max = {rows[0]['top_k']}")
        print(f"  Top-layer summands: {len(rows)}")

        print("  (λ', μ') pairs:")
        for r in rows:
            print(
                f"    λ' = {r['lambda_prime']}, "
                f"μ' = {r['mu_prime']}"
            )

        print("-" * 75)

    if len(counterexamples) > limit:
        print(
            f"\nShowing {limit} of "
            f"{len(counterexamples)} counterexamples."
        )

    return counterexamples


In [91]:
weak_counterexamples = show_weak_counterexamples(results, limit=20)

WEAK STATEMENT — TOP SOCLE LAYER
Partition pairs tested: 38025
Pairs satisfying weak statement: 35775
Counterexamples: 2250
Weak presence rate: 94.08%

Counterexample 1
  λ     = (4, 2)
  μ     = (2, 2, 1, 1)
  β     = (2, 2)
  k_max = 6
  Top-layer summands: 1
  (λ', μ') pairs:
    λ' = (), μ' = ()
---------------------------------------------------------------------------

Counterexample 2
  λ     = (4, 2)
  μ     = (2, 2, 1, 1, 1)
  β     = (2, 2)
  k_max = 6
  Top-layer summands: 1
  (λ', μ') pairs:
    λ' = (), μ' = (1,)
---------------------------------------------------------------------------

Counterexample 3
  λ     = (4, 2)
  μ     = (2, 2, 1, 1, 1, 1)
  β     = (2, 2)
  k_max = 6
  Top-layer summands: 1
  (λ', μ') pairs:
    λ' = (), μ' = (1, 1)
---------------------------------------------------------------------------

Counterexample 4
  λ     = (4, 2)
  μ     = (2, 2, 1, 1, 1, 1, 1)
  β     = (2, 2)
  k_max = 6
  Top-layer summands: 1
  (λ', μ') pairs:
    λ' = (), μ' = 